# Train PatchTST + CVAE on Colab

Run this notebook's kernel connected to a Colab runtime (VS Code: kernel picker top-right -> "Select Another Kernel" -> Google Colab -> pick a GPU runtime).

Run the cells top to bottom. The clone step is idempotent (pulls if already cloned).

In [8]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [9]:
import os

REPO_URL = "https://github.com/WoodyChang21/ECE1508_GenAI.git"
BRANCH = "steven2"

if not os.path.isdir("ECE1508_GenAI"):
    !git clone -b {BRANCH} {REPO_URL}
else:
    # Not a plain `git pull` -- if this clone has ANY local changes (e.g. leftover
    # checkpoints/outputs from an earlier run in the same runtime that never got pushed),
    # a pull can fail outright ("local changes would be overwritten") and Colab just
    # prints the error and moves on -- training then silently proceeds on stale code with
    # no visible failure until much later (e.g. a non-fast-forward push at the end).
    # fetch + hard reset guarantees this checkout exactly matches origin/{BRANCH} no
    # matter what state it was left in.
    !cd ECE1508_GenAI && git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}

%cd ECE1508_GenAI
!git log --oneline -1


Cloning into 'ECE1508_GenAI'...
remote: Enumerating objects: 1636, done.
remote: Counting objects: 100% (1148/1148), done.
remote: Compressing objects: 100% (818/818), done.
remote: Total 1636 (delta 599), reused 861 (delta 328), pack-reused 488 (from 1)
Receiving objects: 100% (1636/1636), 68.16 MiB | 13.18 MiB/s, done.
Resolving deltas: 100% (800/800), done.
/content/ECE1508_GenAI/ECE1508_GenAI
6ed3b31 (HEAD -> steven2, origin/steven2) Revert to random-sample evaluation for hourly models; wire momentum features into both


In [3]:
# torch is preinstalled on Colab; just need mplfinance + pyyaml
!pip install -q mplfinance pyyaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 7.1 MB/s eta 0:00:00


In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## Sanity checks (data pipeline tests)

Cheap to run first -- confirms the feature/window logic before committing to a long training run.

In [5]:
!pip install -q pytest
!python -m pytest steven/tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/ECE1508_GenAI
plugins: anyio-4.14.2, typeguard-4.5.2, langsmith-0.10.2
collected 47 items                                                             

steven/tests/test_cvae_inpainting.py::test_default_decoder_ctx_dim_matches_old_behavior PASSED [  2%]
steven/tests/test_cvae_inpainting.py::test_decoder_ctx_dim_bottlenecks_decoder_input PASSED [  4%]
steven/tests/test_cvae_inpainting.py::test_prior_head_still_sees_full_ctx_dim_when_bottlenecked PASSED [  6%]
steven/tests/test_data_pipeline.py::test_reconstruct_prices_round_trip PASSED [  8%]
steven/tests/test_data_pipeline.py::test_anchor_correction_matches_close_0_for_all_horizon_bars PASSED [ 10%]
steven/tests/test_data_pipeline.py::test_wick_components_non_negative PASSED [ 12%]
steven/tests/test_data_pipeline.py::test_build_window_

## Momentum feature setup (EMA9/EMA21 + RSI-14 + VIX)

Both models below train on the base hourly OHLCV plus three added features (`src/momentum_pipeline.py`)
-- EMA9/EMA21 crossover, RSI-14, and VIX (previous trading day's close, a genuine market-derived
sentiment signal, not a transform of SPY's own price). VIX isn't in the hourly parquet, so it's pulled
fresh here via `yfinance` before training. Fresh pull each run rather than committing the parquet --
cheap (a few seconds), avoids the data going stale, matches how the rest of this notebook already
re-clones/re-installs fresh each time.

In [10]:
!pip install -q -r steven/requirements-probe.txt
!python steven/src/collect_vix_yfinance.py

20:14:40 wrote 8139 VIX daily bars (1993-01-29 to 2025-05-29) to /content/ECE1508_GenAI/ECE1508_GenAI/steven/data/vix_daily_yfinance.parquet


## Train PatchTST (real defaults: 20k windows/epoch, 20 epochs, configs/patchtst_hourly_momentum.yaml)

Same architecture/training recipe as `configs/patchtst.yaml`, plus the momentum features from the setup
cell above (`PatchTST` gained an `n_feature_channels` param for this -- see `src/models/patchtst.py`).
Drop `--max-epochs`/`--train-windows-per-epoch` overrides below if you want a quick smoke run first
instead of the full config.

In [11]:
!python steven/src/train_patchtst.py --config steven/configs/patchtst_hourly_momentum.yaml --device auto

20:14:49 device: cuda
20:14:49 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
20:14:49 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
20:14:49 momentum features enabled: ema_cross/trend_position/rsi/vix, N_FEATURE_CHANNELS=11
20:14:54 epoch 1/20  train_loss=0.23837  val_loss=0.13126  (2.6s)
20:14:54   -> saved best checkpoint (val_loss=0.13126) to steven/outputs/patchtst_checkpoint.pt
20:14:56 epoch 2/20  train_loss=0.16639  val_loss=0.11737  (2.1s)
20:14:56   -> saved best checkpoint (val_loss=0.11737) to steven/outputs/patchtst_checkpoint.pt
20:14:58 epoch 3/20  train_loss=0.15368  val_loss=0.10626  (2.1s)
20:14:58   -> saved best checkpoint (val_loss=0.10626) to steven/outputs/patchtst_checkpoint.pt
20:15:00 epoch 4/20  train_loss=0.14402  val_loss=0.11179  (2.2s)
20:15:02 epoch 5/20  train_loss=0.13872  val_loss=0.10042  (2.1s)
20:15:02   -> saved best checkpoint (val_loss=0.10042) to steven/outputs/patchtst_checkpoint.pt
2

## Train CVAE (real defaults: 20k windows/epoch, 30 epochs, configs/cvae_hourly_momentum.yaml)

Same architecture/loss recipe as `configs/cvae.yaml` (z_dim=8, decoder_ctx_dim=8, price_scale,
w_direction -- unchanged, extensively tested and never found to be the bottleneck), plus the momentum
features from the setup cell above.

In [12]:
!python steven/src/train_cvae.py --config steven/configs/cvae_hourly_momentum.yaml --device auto

20:15:37 device: cuda
20:15:37 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
20:15:37 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
20:15:37 momentum features enabled: ema_cross/trend_position/rsi/vix, N_CHANNELS=13
20:15:38 price_scale (open_ret, body_ret, upper_wick, lower_wick): [0.002682909369468689, 0.0028937843162566423, 0.0016679799882695079, 0.0017456382047384977]
20:15:41 epoch 1/30  beta=0.20  train_recon=398.30625 (kl=21.2622 dir=2.6030)  val_recon=8.88281 (kl=16.1238 dir=1.7098)  (2.2s)
20:15:41   -> saved best checkpoint (val_recon=8.88281) to steven/outputs/cvae_checkpoint.pt
20:15:43 epoch 2/30  beta=0.40  train_recon=6.99316 (kl=8.3604 dir=1.2490)  val_recon=5.13879 (kl=5.7030 dir=0.9329)  (1.6s)
20:15:43   -> saved best checkpoint (val_recon=5.13879) to steven/outputs/cvae_checkpoint.pt
20:15:44 epoch 3/30  beta=0.60  train_recon=4.68079 (kl=3.9846 dir=0.9466)  val_recon=4.27858 (kl=2.4375 dir=0.8665)  (1.6s)

## Daily CVAE (EMA9/EMA21 + RSI-14 + VIX, rolling-window training) -- consolidated build

Replaces the earlier context-length-sweep probe (`probe_daily_cvae.py`, obsolete -- superseded
by everything below). This is the current best-tested daily-bars recipe, consolidated into one
build after extensive investigation (see `cvae_direction_collapse.md`'s "momentum enrichment",
"Adding VIX", and "robustness check" sections for the full history):

- **Data**: SPY's full 1993-2025 daily history pulled fresh via `yfinance`
  (`src/collect_daily_yfinance.py`) -- not a resample of the hourly parquet, which would only
  reach back to 2010. VIX pulled the same way (`src/collect_vix_yfinance.py`) as a genuine
  market-derived sentiment feature, not a transform of SPY's own price.
- **Features**: the same 4 price/wick components + volume as the hourly project, plus
  EMA9/EMA21 crossover, RSI-14, and VIX (previous trading day's close, so it's genuinely
  known before that day's open -- see `src/momentum_pipeline.py`).
- **Context**: 10 trading days -- matched calendar lookback vs. the hourly model's 70 bars
  (not bar count), sampled with a rolling window (every valid window used exactly once per
  epoch, not `WindowSampler`'s random multi-length draw -- see `probe_momentum_rolling_cvae.py`'s
  module docstring for why).
- **Architecture/loss**: unchanged from `configs/cvae.yaml`'s hourly recipe (z_dim=8,
  decoder_ctx_dim=8, price_scale, w_direction) -- extensively tested and never found to be the
  bottleneck, so kept constant here for comparability rather than re-tuned.
- **Trading strategy**: the same bracket take-profit/stop-loss walk-forward as the hourly
  project, but `stop_loss_pct` and `min_return_threshold` are both re-derived from daily's own
  measured volatility (`sell_bound`, the p99 3-day anchored move) instead of reusing the two
  hourly-tuned constants -- printed in this run's own log/report below.

**Headline finding going in, so the result below isn't a surprise**: a robustness check across
5 training seeds found CVAE's correlation with real daily market direction is statistically
indistinguishable from zero (mean +0.09, std 0.16 -- individual runs have landed anywhere from
-0.17 to +0.29 purely from training randomness). This build won't manufacture a real edge that
isn't there; it's still worth training and inspecting directly.

Writes to `cvae_checkpoint_momentum_rolling_daily.pt` -- does not touch or overwrite the hourly
`cvae_checkpoint.pt` above.

In [ ]:
# yfinance isn't needed by the main hourly pipeline -- kept in its own requirements file
# (steven/requirements-probe.txt) rather than requirements-model.txt.

# !pip install -q -r steven/requirements-probe.txt

# Fresh pull each run rather than committing the parquets -- cheap (a few seconds), avoids
# the data going stale relative to whatever TEST_END this project is using, and matches how
# the rest of this notebook already re-clones/re-installs fresh each time.

# !python steven/src/collect_daily_yfinance.py
# !python steven/src/collect_vix_yfinance.py

19:30:47 wrote 8139 daily bars (1993-01-29 to 2025-05-29) to /content/ECE1508_GenAI/steven/data/spy_daily_yfinance.parquet
19:30:48 wrote 8139 VIX daily bars (1993-01-29 to 2025-05-29) to /content/ECE1508_GenAI/steven/data/vix_daily_yfinance.parquet


In [ ]:
# import sys
# sys.path.insert(0, "steven")
# import probe_momentum_rolling_cvae as pmr
# from IPython.display import Markdown, display

# results = pmr.main("daily")
# display(Markdown(pmr.format_report(results)))


=== body_ret variance/correlation, N=300 windows, k=5 ===
  ratio (>1 = context beats sampling noise): 0.662
  correlation with 10-bar trend: -0.0435

=== predicted_return (take_profit vs close_0), N=300 windows ===
  mean: +0.1208%  pct eligible (>0): 100.0%
  percentiles 5/25/50/75/95: +0.0660% / +0.1031% / +0.1262% / +0.1417% / +0.1681%


### daily CVAE, ctx_bars=10, test 2023-01-01 to 2025-05-30

- stop_loss_pct=0.0515, min_return_threshold=0.00052 (sell_bound=0.0515)
- direction diagnostic (N=300 test windows): variance ratio=0.662, correlation with trend=-0.0435

| | CVAE | buy&hold | naive_periodic |
|---|---|---|---|
| n_trades / n_decisions | 196/201 | -- | 197 |
| total_return | +3.43% | +48.34% | +30.68% |
| win_rate | 88.3% | -- | 56.9% |
| take_profit_rate | 87.8% | -- | -- |
| avg_return per trade | +0.0202% | -- | +0.1434% |

**outcome breakdown (fraction of all decisions):**

| win_take_profit | win_expiry | lose_expiry | lose_stop_loss | skipped | no_trade |
|---|---|---|---|---|---|
| 85.6% | 0.5% | 10.9% | 0.5% | 2.5% | 0.0% |

## Evaluate both models on the fixed test set

Random-sample backtest (not walk-forward -- see `cvae_direction_collapse.md`'s "revisit the
pre-walk-forward era" discussion for why this switched back): draws 100 independently-sampled test
windows across all 9 context lengths, evaluates both models on the identical drawn set, no
confidence-threshold sweep, no buy-and-hold comparison (these windows can overlap in calendar time, so
there's no real equity curve to compound -- see `src/evaluate.py`'s module docstring). Momentum features
are auto-detected from each checkpoint's own saved config, so this cell needs no changes regardless of
which config trained the checkpoints on disk.

In [13]:
!python steven/src/evaluate.py \
  --patchtst-checkpoint steven/outputs/patchtst_checkpoint.pt \
  --cvae-checkpoint steven/outputs/cvae_checkpoint.pt \
  --device auto

20:16:42 device: cuda
20:16:42 Gap report: 145 fully missing weekdays, 39 short sessions (<7 bars)
20:16:42 Dropping first row (2010-01-04 09:30:00): no previous close to compute a return from
20:16:42 momentum features enabled: ema_cross/trend_position/rsi/vix, N_CHANNELS=13
20:16:42 sell-price shrink bound: p99.0 of |anchored log return| over train = 0.0190 (vs. model's own MAX_LOG_RETURN)
20:16:42 evaluating on 100 randomly-sampled test windows (spread across 9 context lengths, no walk-forward, no confidence sweep -- patchtst_min_return>=0.100%, cvae_min_return>=0.020% [no CVAE quality gate -- see backlog.md], stop_loss=2.00% [shared, whichever hits first, stop-loss wins same-bar ties -- see bracket_exit])...
20:16:43 random-sample: PatchTST 13 trades / 100 decisions, CVAE 0 trades / 100 decisions
20:16:43 wrote metrics to steven/outputs/metrics.json
20:16:43 random_sample: {
  "n_test_windows": 100,
  "patchtst_min_return_threshold": 0.001,
  "cvae_min_return_threshold": 0.0002,
  

## Refresh v1.md from this run

Rewrites the Results/backtest tables and sample images in `steven/v1.md` from the metrics.json + sample_plots this run just produced (see `steven/src/update_report.py`). Only the tables/images are rewritten -- surrounding prose (interpretation, caveats) is left as-is; review it by hand if the story changed. This only edits the file in the cloned repo here -- push/download separately if you want to keep it.

In [14]:
!python steven/src/update_report.py

20:17:03 updated steven/v1.md: results-samples, hit-summary, spread-summary, random-sample-strategies, random-sample-outcome-breakdown
20:17:03 not auto-updated -- reread and edit by hand if the story changed: the historical narrative sections (Loss, Bounding, Trading criteria's stop-loss/quality-gate history, Strengths/weaknesses/next-steps) describe how the project got here and are left as written; only the sections that directly describe the CURRENT evaluation methodology (Results, Random-sample backtest, Key terms' Backtest definition, Workflow step 5, How to reproduce step 3) were updated for the switch away from walk-forward.


## Generative-quality CVAE: NLL + learned variance (configs/cvae_generative.yaml)

This project's actual goal is generating diverse, plausible, context-appropriate future candles -- not trading (see `cvae_direction_collapse.md`'s "generative pivot" discussion). The cells above train/evaluate the trading-framed checkpoints and stay as-is; this section is additive, not a replacement.

`configs/cvae_generative.yaml` is the same architecture as `configs/cvae_hourly_momentum.yaml` (momentum features on, `z_dim=8`/`ctx_dropout=0.3`/`decoder_ctx_dim=8`), but the decoder now predicts a per-component variance and trains against a real NLL (Laplace on open_ret/body_ret, Gaussian on wicks/volume) instead of plain MSE, with a 5-epoch mean-only warmup to avoid the classic "inflate variance instead of improving the mean" pathology. `w_direction` is disabled (0.0) -- it never beat chance at its own trading-motivated goal and has no generative-quality justification.

In [ ]:
!python steven/src/train_cvae.py --config steven/configs/cvae_generative.yaml --device auto

## Evaluate generative quality: diversity, calibration, context-sensitivity

Runs `src/evaluate_generative.py` over a full deterministic rolling-window pass (default `ctx_bars=70`, `k=32` samples/window) and reports, in model-native (log-return/z-scored) units:
- **Diversity** -- are a window's k sampled draws meaningfully different from each other?
- **Calibration** -- CRPS + rank histogram from the k samples directly, plus PIT/coverage curves using the learned variance (only for a `reconstruction: nll` checkpoint).
- **Context-sensitivity** -- the property this pivot is actually about: does generated output shift across realized-volatility regimes the way real data does (`effect_ratio` ≈ 1 is good, ≈ 0 means context-blind generation), and does conditioning on context beat a context-blind climatology baseline at all (`crps_skill_score`)?

Writes `steven/outputs/generative_metrics.json` and two plots to `steven/outputs/generative_plots/`: a regime x k-samples grid (rows = low/mid/high volatility buckets, columns = ground truth + generated draws) and a diversity fan chart (k sampled close-price paths overlaid on one context window).

In [ ]:
!python steven/src/evaluate_generative.py \
  --cvae-checkpoint steven/outputs/cvae_checkpoint_generative.pt \
  --device auto

## Sync results back to GitHub

Commits `steven/outputs/` (checkpoints, metrics.json, sample_plots) and the regenerated `steven/v1.md` from this Colab runtime and pushes straight to the `steven2` branch -- no manual zip/download step. That step wasn't reliably reaching the local machine: `files.download()`'s browser-download trick only works from the Colab web UI, not when this kernel is attached remotely (e.g. from VS Code's kernel picker), so nothing ever landed on disk.

Needs a GitHub personal access token with `repo` write scope for this push only -- entered via `getpass` below, never written to the notebook or committed anywhere.

In [41]:
# %%bash
# git fetch origin steven2
# git merge origin/steven2 --no-edit

In [15]:
import getpass

token = getpass.getpass("GitHub PAT (repo write, used only for this push): ")

In [16]:
%%bash -s "$token"
TOKEN="$1"
if [ -z "$TOKEN" ]; then
  echo "Token was empty -- re-run the getpass cell above and actually paste your PAT before pressing Enter." >&2
  exit 1
fi
git config user.email "colab@ephemeral.local"
git config user.name "Colab Runtime"
git add steven/outputs steven/v1.md
if git diff --cached --quiet; then
  echo "Nothing new to commit -- outputs/v1.md unchanged from last commit."
else
  git commit -m "Retrain + refresh results from Colab run"
fi
# Push unconditionally -- a prior run may have committed but failed to push (e.g. a blank
# token), in which case there's nothing new to commit here but HEAD is still ahead of origin.
# Pushes to steven2, not steven -- this notebook and this branch are the working copy for
# now; steven is left alone so a second person's in-flight work there can't collide with
# this runtime's pushes.
git push "https://${TOKEN}@github.com/WoodyChang21/ECE1508_GenAI.git" HEAD:steven2

[steven2 2029faa] Retrain + refresh results from Colab run
 6 files changed, 49 insertions(+), 176 deletions(-)
 rewrite steven/outputs/cvae_checkpoint.pt (92%)
 rewrite steven/outputs/metrics.json (77%)
 rewrite steven/outputs/patchtst_checkpoint.pt (92%)
 delete mode 100644 steven/outputs/sample_plots/no_trade_start26557_ctx70.png
 rewrite steven/outputs/sample_plots/samples.json (100%)


To https://github.com/WoodyChang21/ECE1508_GenAI.git
   6ed3b31..2029faa  HEAD -> steven2


### Fallback: zip + browser download

Only useful if you're running this notebook inside the actual Colab web UI (not a remote kernel) and would rather download a zip than push through git.

In [33]:
!zip -r outputs.zip steven/outputs

try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Not in a Colab frontend session -- outputs.zip is in the working dir, grab it manually.")

  adding: steven/outputs/ (stored 0%)
  adding: steven/outputs/metrics.json (deflated 68%)
  adding: steven/outputs/cvae_checkpoint.pt (deflated 8%)
  adding: steven/outputs/sample_plots/ (stored 0%)
  adding: steven/outputs/sample_plots/samples.json (deflated 69%)
  adding: steven/outputs/sample_plots/no_trade_start26500_ctx70.png (deflated 10%)
  adding: steven/outputs/patchtst_checkpoint.pt (deflated 9%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Optional: reproduce the pre-fix CVAE checkpoint (comparison experiment)

Commit `2c4ad99` (before `price_scale`, `decoder_ctx_dim`, or `w_direction` existed) is the
best walk-forward result CVAE has produced (+6.79% total return, 66.4% trade rate). Every fix
attempted since has made the loss more "correct" by the direction-collapse diagnosis but
hasn't matched it. `configs/cvae_pre_fix_repro.yaml` reproduces that exact config (writes to
a separate checkpoint so it doesn't clobber `configs/cvae.yaml`'s), and
`src/diagnose_cvae_direction.py` reruns the same variance/correlation/eligibility checks used
throughout `cvae_direction_collapse.md` against any checkpoint -- run it against both to see
whether the fixes actually helped or whether `2c4ad99` was a favorable roll against one test
path. See `cvae_direction_collapse.md`'s "Revisiting the pre-collapse-chasing checkpoint".

In [ ]:
!python steven/src/train_cvae.py --config steven/configs/cvae_pre_fix_repro.yaml --device auto

In [ ]:
print("=== current checkpoint (price_scale + decoder_ctx_dim + w_direction) ===")
!python steven/src/diagnose_cvae_direction.py --cvae-checkpoint steven/outputs/cvae_checkpoint.pt

print("\n=== pre-fix repro checkpoint (matches commit 2c4ad99) ===")
!python steven/src/diagnose_cvae_direction.py --cvae-checkpoint steven/outputs/cvae_checkpoint_pre_fix_repro.pt